### package

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats
import pandas as pd
from scipy.stats import linregress
import cmaps

import cartopy.crs as ccrs
import cartopy.mpl.ticker as cticker
from cartopy.util import add_cyclic_point

In [ ]:
cmap = cmaps.GMT_panoply

# Data

In [ ]:
#input fishery data

df_PBF = pd.read_csv('recruitment.csv')
df_PBF_lag = df_PBF.loc[(df_PBF['year'] >= 1981) & (df_PBF['year'] <= 2020)] #lag1
#加入量對數轉換
log_recruitment=np.log10(df_PBF_lag['recruitment'])

In [ ]:
#input environmwntal data and select area
url1 = 'sst.mnmean1.nc'
ds_sst = xr.open_dataset(url1)
#ds_sst_select_yy_area=ds_sst.sel(time=slice('1981','2020'),lat=slice(50,15),lon=slice(115,150))
ds_sst_select_yy_area=ds_sst.sel(time=slice('1981','2020'),lat=slice('90','0'),lon=slice(100,355))#常用Fig1、Fig4

# 選取需要的月份

In [ ]:
#choose select month to average
ds_sst_select_yy_area['time'] = pd.to_datetime(ds_sst_select_yy_area['time'])
month_mask = ds_sst_select_yy_area['time.month'].isin([10,11,12 ])
#month_mask = ds_sst_select_yy_area['time.month'].isin([ 4,5,6])
#month_mask = ds_sst_select_yy_area['time.month'].isin([ 7,8,9])
#month_mask = ds_sst_select_yy_area['time.month'].isin([ 7,8,9,10,11,12])
ds_sst_select_yy_area_mm = ds_sst_select_yy_area.where(month_mask, drop=True)  
ds_sst_yearly_mean = ds_sst_select_yy_area_mm.groupby('time.year').mean('time')   
#ds_sst_yearly_mean1=ds_sst_yearly_mean.mean(dim='year') 
ds_sst_yearly_mean

# 計算相關性與檢定

In [ ]:
nx=len(ds_sst_yearly_mean['lon']) #raw
ny=len(ds_sst_yearly_mean['lat']) #raw
# nx=len(ds_sst_yearly_mean['longitude']) #raw
# ny=len(ds_sst_yearly_mean['latitude']) #raw

p_array=np.zeros((ny,nx))
r_array=np.zeros((ny,nx))
m_array=np.zeros((ny,nx))

for i in range(nx):
    for j in range(ny):
        
        x=df_PBF_lag['obs_PO_mode2']#nolag 
        #x=df_PBF['OND_PC1']#nolag
        #x=log_recruitment #nolag 
        #x=df_PBF_lag['Recruitment ']
        y=ds_sst_yearly_mean['sst'][:,j,i] #raw
        
        m,b,r,p,e = linregress(x,y)
        
        m_array[j,i]=m
        r_array[j,i]=r
        p_array[j,i]=p

In [ ]:
mask_sig=np.where(p_array<0.05, r_array, np.nan)

# 只畫過檢定的

# 通通都畫

In [ ]:
 a = 'PO'  
 b = 2
 mode_col = f'{a} Mode{b}'
#plotting
clevs=np.arange(-0.8,0.81,0.1)
#clevs=np.arange(14,28,1)

fig = plt.figure(figsize=(14,10))

# Set the axes using the specified map projection
ax=plt.axes(projection=ccrs.PlateCarree(central_longitude=225))#225

# Add cyclic point
data=r_array
data,lon=add_cyclic_point(data,coord=ds_sst_yearly_mean['lon'])
mask_data,lons=add_cyclic_point(mask_sig,coord=ds_sst_yearly_mean['lon'])

# Make a filled contour plot
cs=ax.contourf(lon,ds_sst_yearly_mean['lat'],
            data,clevs,
            transform=ccrs.PlateCarree(),
            cmap='coolwarm',extend='both') #bwr#jet#coolwarm

ax.contourf(lon,ds_sst_yearly_mean['lat'],mask_data,[0,1],
            transform = ccrs.PlateCarree(),colors='none',
            hatches=['.','.'],extend='both',alpha=0)

# data=r_array
# data,lon=add_cyclic_point(data,coord=ds_sst_yearly_mean['longitude'])
# mask_data,lons=add_cyclic_point(mask_sig,coord=ds_sst_yearly_mean['longitude'])

# # Make a filled contour plot
# cs=ax.contourf(lon,ds_sst_yearly_mean['latitude'],
#             data,clevs,
#             transform=ccrs.PlateCarree(),
#             cmap='coolwarm',extend='both') #bwr#jet#coolwarm

# ax.contourf(lon,ds_sst_yearly_mean['latitude'],mask_data,[0,1],
#             transform = ccrs.PlateCarree(),colors='none',
#             hatches=['.','.'],extend='both',alpha=0)

# Add coastlines
ax.coastlines()

# Add gridlines
ax.gridlines()

# Define the xticks for longtitude
ax.set_xticks(np.arange(100,355,45),crs=ccrs.PlateCarree())
#ax.set_xticks(np.arange(-180,180,60),crs=ccrs.PlateCarree())
#ax.set_xticks(np.arange(115,151,5),crs=ccrs.PlateCarree())
#ax.set_xticks(np.arange(125,146,5),crs=ccrs.PlateCarree())
lon_formatter=cticker.LongitudeFormatter()
ax.xaxis.set_major_formatter(lon_formatter)
ax.tick_params(axis='x', labelsize=30)

# Define ytick for latitude(-90,91,30)
ax.set_yticks(np.arange(15,105,15),crs=ccrs.PlateCarree())
#ax.set_yticks(np.arange(-90,91,30),crs=ccrs.PlateCarree())
#ax.set_yticks(np.arange(15,90,15),crs=ccrs.PlateCarree())
#ax.set_yticks(np.arange(15,51,5),crs=ccrs.PlateCarree())
#ax.set_yticks(np.arange(25,46,5),crs=ccrs.PlateCarree())
lat_formatter=cticker.LatitudeFormatter()
ax.yaxis.set_major_formatter(lat_formatter)
ax.tick_params(axis='y', labelsize=30) 

# Call colorbar
cbar=plt.colorbar(cs,orientation='horizontal',shrink=0.8,
                 label='Correlation Coefficient')
cbar.ax.tick_params(labelsize=24)
cbar.ax.set_xlabel('Correlation Coefficient ', fontsize=24)
# Add title

plt.title(f'{mode_col} vs SST ',fontsize=28, loc='left', pad=16)
plt.savefig(f'{mode_col}_vs_SST_ano.svg', dpi=300)

In [ ]:
#聚焦日本海域
# a = 'AO'  
# b = 3
# mode_col = f'{a}_mode{b}'
#plotting
clevs=np.arange(-0.8,0.81,0.1)
#clevs=np.arange(14,28,1)

fig = plt.figure(figsize=(14,10))

# Set the axes using the specified map projection
ax=plt.axes(projection=ccrs.PlateCarree(central_longitude=180))#225

# Add cyclic point
data=r_array
data,lon=add_cyclic_point(data,coord=ds_sst_yearly_mean['lon'])
mask_data,lons=add_cyclic_point(mask_sig,coord=ds_sst_yearly_mean['lon'])

# Make a filled contour plot
cs=ax.contourf(lon,ds_sst_yearly_mean['lat'],
            data,clevs,
            transform=ccrs.PlateCarree(),
            cmap='coolwarm',extend='both') #bwr#jet#coolwarm

ax.contourf(lon,ds_sst_yearly_mean['lat'],mask_data,[0,1],
            transform = ccrs.PlateCarree(),colors='none',
            hatches=['.','.'],extend='both',alpha=0)

# Add coastlines
ax.coastlines()

# Add gridlines
ax.gridlines()
ax.set_extent([120, 150, 16, 50], crs=ccrs.PlateCarree())
# Define the xticks for longtitude
#ax.set_xticks(np.arange(100,355,45),crs=ccrs.PlateCarree())
#ax.set_xticks(np.arange(-180,180,60),crs=ccrs.PlateCarree())
ax.set_xticks(np.arange(120,151,10),crs=ccrs.PlateCarree())
#ax.set_xticks(np.arange(125,146,5),crs=ccrs.PlateCarree())
lon_formatter=cticker.LongitudeFormatter()
ax.xaxis.set_major_formatter(lon_formatter)
ax.tick_params(axis='x', labelsize=30, pad=15)

# Define ytick for latitude(-90,91,30)
#ax.set_yticks(np.arange(-90,91,30),crs=ccrs.PlateCarree())
#ax.set_yticks(np.arange(15,90,15),crs=ccrs.PlateCarree())
#ax.set_yticks(np.arange(15,51,5),crs=ccrs.PlateCarree())
ax.set_yticks(np.arange(20,51,5),crs=ccrs.PlateCarree())
lat_formatter=cticker.LatitudeFormatter()
ax.yaxis.set_major_formatter(lat_formatter)
ax.tick_params(axis='y', labelsize=30) 

# Call colorbar
cbar=plt.colorbar(cs,orientation='horizontal',shrink=0.8,
                 label='Correlation Coefficient')
cbar.ax.tick_params(labelsize=24)
cbar.ax.set_xlabel('Correlation Coefficient ', fontsize=24)
# Add title

plt.title('OND_EOF1',fontsize=28, loc='left', pad=16)#(c) Correlation between log recruitment and SST (AMJ)#Juvenile nursery habitat
plt.savefig(f'{mode_col}_vs_SST_ano.svg', dpi=300)#final fig1